# Train YOLO26 semantic segmentation on Colab

Mirrors `scripts/train.py` — same `src/training.py` code path as local training, just running on Colab's free T4 GPU instead of an M-series MPS backend.

Free-tier Colab sessions can disconnect without warning (session cap ~12h, weekly GPU quota), so this notebook mounts Google Drive and writes checkpoints there as training runs, not just at the end.

**Before running:** get this project onto the Colab VM (see the two options in the next cell), and get your dataset into Drive — either the `datasets/yolo_datasets/<region>/ultralytics/` folder built locally by `main.py build-dataset`, or a COCO export zip (from `main.py build-dataset --export-format coco`, or downloaded from Roboflow).

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Get the project onto this VM

**Option A — GitHub** (if you've pushed this repo): uncomment and fill in the URL.

**Option B — Drive** (default here, since there's no remote configured yet): keep a copy of the repo in Drive (e.g. via Google Drive's desktop sync, or by uploading a zip once) and point `PROJECT_DIR` at it.

In [ ]:
# Option A — clone from GitHub, if you've pushed this repo there:
# !git clone https://github.com/<you>/coastal-recession-ml.git /content/coastal-recession-ml
# PROJECT_DIR = "/content/coastal-recession-ml"

# Option B — repo already sitting in Drive:
PROJECT_DIR = "/content/drive/MyDrive/coastal-recession-ml"

%cd {PROJECT_DIR}

In [ ]:
# Colab already ships a CUDA-enabled torch — requirements.txt deliberately doesn't
# pin torch, so this won't clobber it with an incompatible build.
!pip install -q -r requirements.txt

## Choose your dataset source

Set `DATA_SOURCE` to `"local"` if you're pointing at an already-built `datasets/yolo_datasets/<region>/ultralytics/` folder (e.g. synced via Drive), or `"coco"` if you have a COCO-format zip (our own `coco_export.py` output, or a Roboflow export) to convert first.

In [ ]:
DATA_SOURCE = "local"  # "local" or "coco"

# --- if DATA_SOURCE == "local" ---
LOCAL_DATA_YAML = f"{PROJECT_DIR}/datasets/yolo_datasets/mediterranean/ultralytics/data.yaml"

# --- if DATA_SOURCE == "coco" ---
COCO_ZIP_PATH = "/content/drive/MyDrive/coastal-recession-ml/roboflow_export.zip"
COCO_EXTRACT_DIR = "/content/coco_export"
COCO_OUTPUT_DIR = "/content/drive/MyDrive/coastal-recession-ml/converted_coco_dataset"

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

if DATA_SOURCE == "coco":
    import shutil
    shutil.unpack_archive(COCO_ZIP_PATH, COCO_EXTRACT_DIR)

    from scripts.prepare_coco_dataset import find_split_dir, SPLIT_DIR_CANDIDATES
    from src.coco_to_masks import convert_coco_split
    from src.semantic_export import write_data_yaml
    from pathlib import Path

    input_dir = Path(COCO_EXTRACT_DIR)
    output_dir = Path(COCO_OUTPUT_DIR)
    train_dir = find_split_dir(input_dir, SPLIT_DIR_CANDIDATES["train"])
    val_dir = find_split_dir(input_dir, SPLIT_DIR_CANDIDATES["val"])

    class_names = None
    for split, split_dir in [("train", train_dir), ("val", val_dir)]:
        names = convert_coco_split(
            coco_json_path=split_dir / "_annotations.coco.json",
            images_dir=split_dir,
            output_images_dir=output_dir / "images" / split,
            output_masks_dir=output_dir / "masks" / split,
        )
        print(f"{split}: classes = {names}")
        class_names = class_names or names

    write_data_yaml(output_dir, class_names)
    DATA_YAML = str(output_dir / "data.yaml")
else:
    DATA_YAML = LOCAL_DATA_YAML

print("Training on:", DATA_YAML)

## Train

`project=` points into Drive, so checkpoints (`last.pt` after every epoch, `best.pt`) survive a dropped session — resume with `model=".../weights/last.pt"` if it disconnects mid-run.

In [ ]:
from src.training import train_model

results = train_model(
    data_yaml=DATA_YAML,
    model="yolo26n-sem.pt",
    epochs=100,
    imgsz=640,
    batch=16,
    device=None,  # auto-detects the Colab GPU
    seed=42,
    project="/content/drive/MyDrive/coastal-recession-ml/runs/train",
    name="coastal-seg",
)